In [21]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from pyspark.sql.window import *
from notebookutils import *
from pyspark.sql import functions as F
from delta.tables import *
from notebookutils import mssparkutils

StatementMeta(, b7cd0d21-3cd8-4a68-a843-ccf8bbd68db2, 23, Finished, Available, Finished, False)

In [22]:
tables_str = tables_str
bronze = bronze
silver = silver
workspace = workspace
lakehouse = lakehouse
silver = silver

# tables_str = "Patient,Encounter,Observation,Condition"
# bronze = "bronze"
# silver = "silver"
# workspace = "Fabric_Assignment_Synapx"
# lakehouse = "FHIR_Lakehouse"

Tables = [t.strip() for t in tables_str.split(",")]

StatementMeta(, b7cd0d21-3cd8-4a68-a843-ccf8bbd68db2, 24, Finished, Available, Finished, False)

In [23]:
table = None
for i in Tables:
    if i.lower() == "patient":
        table = i
        break

if table is None:
    mssparkutils.notebook.exit("Exiting notebook: Table is not patient")

print(table + " table is present ")

StatementMeta(, b7cd0d21-3cd8-4a68-a843-ccf8bbd68db2, 25, Finished, Available, Finished, False)

Patient table is present 


In [24]:
bronze_path = f"Files/{bronze}/{table.lower()}"
silver_path = f"Files/{silver}/{table.lower()}"

# Check if silver path exists
silver_exists = mssparkutils.fs.exists(silver_path)

# Read bronze data
df = spark.read.format("delta").load(bronze_path)

# Apply filter only if silver exists
if silver_exists:
    df = df.filter(
        col("load_date") >= date_sub(current_date(), 4)
    )

df.printSchema()


StatementMeta(, b7cd0d21-3cd8-4a68-a843-ccf8bbd68db2, 26, Finished, Available, Finished, False)

root
 |-- api_url: string (nullable = true)
 |-- extraction_timestamp: string (nullable = true)
 |-- load_date: string (nullable = true)
 |-- raw_json: struct (nullable = true)
 |    |-- entry: string (nullable = true)
 |    |-- id: string (nullable = true)
 |    |-- link: string (nullable = true)
 |    |-- meta: string (nullable = true)
 |    |-- resourceType: string (nullable = true)
 |    |-- total: string (nullable = true)
 |    |-- type: string (nullable = true)



In [25]:

df_flat = (
    df
    # ----------------------------
    # Bundle-level / entry-level extraction
    # ----------------------------
    .withColumn("patient_id", regexp_extract(col("raw_json.entry"), r"id=(\d+)", 1))
    .withColumn("gender", regexp_extract(col("raw_json.entry"), r"gender=([a-zA-Z]+)", 1))
    .withColumn("fullUrl", regexp_extract(col("raw_json.entry"), r"fullUrl=([^,}]+)", 1))
    .withColumn("family_name", regexp_extract(col("raw_json.entry"), r"family=([^,}]+)", 1))

    # ----------------------------
    # FIXED given_name
    # ----------------------------
    .withColumn(
        "given_name",
        regexp_extract(
            col("raw_json.entry"),
            r"given=\[([^\]]*)\]",
            1
        )
    )

    # ----------------------------
    # NEW: last_updated (FHIR meta.lastUpdated)
    # Example: lastUpdated=2026-04-13T00:22:52.054+00:00
    # ----------------------------
    .withColumn(
        "last_updated",
        regexp_extract(
            col("raw_json.entry"),
            r"lastUpdated=([^,}]+)",
            1
        )
    )

    # ----------------------------
    # Final output
    # ----------------------------
    .select(
        "extraction_timestamp",
        "load_date",
        "patient_id",
        "gender",
        "family_name",
        "given_name",
        "last_updated"
    )
)

display(df_flat)
silver_condition = df_flat

StatementMeta(, b7cd0d21-3cd8-4a68-a843-ccf8bbd68db2, 27, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bc37988e-4402-4345-9c51-67d573564155)

In [26]:
# -----------------------------
# Step 1: Ensure timestamp type
# -----------------------------
silver_condition = silver_condition.withColumn(
    "last_updated",
    to_timestamp(col("last_updated"))
)

# -----------------------------
# Step 2: Define window
# -----------------------------
window = Window.partitionBy("patient_id").orderBy(
    col("last_updated").desc()
)

# -----------------------------
# Step 3: Deduplicate using row_number
# -----------------------------
silver_condition = (
    silver_condition
    .withColumn("rn", row_number().over(window))
    .filter(col("rn") == 1)
    .drop("rn")
)

# -----------------------------
# Step 4: Display result (Fabric-safe)
# -----------------------------
display(silver_condition.limit(5))

StatementMeta(, b7cd0d21-3cd8-4a68-a843-ccf8bbd68db2, 28, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 142958ea-5a9e-4fcb-a250-8bf7d23a23a9)

In [27]:
staging_df = silver_condition

staging_df = staging_df \
    .withColumn("effective_start_date", col("last_updated").cast("timestamp")) \
    .withColumn("effective_end_date", lit(None).cast("timestamp")) \
    .withColumn("is_current", lit(True))

StatementMeta(, b7cd0d21-3cd8-4a68-a843-ccf8bbd68db2, 29, Finished, Available, Finished, False)

In [28]:
silver_path = f"Files/{silver}/{table.lower()}"

if not mssparkutils.fs.exists(silver_path):
    staging_df.write.format("delta").mode("overwrite").save(silver_path)

    mssparkutils.notebook.exit("Exiting notebook: First time load completed")
else:
    print("Table exists")

StatementMeta(, b7cd0d21-3cd8-4a68-a843-ccf8bbd68db2, 30, Finished, Available, Finished, False)

Table exists


In [29]:
staging_df.createOrReplaceTempView("staging_patient")

StatementMeta(, b7cd0d21-3cd8-4a68-a843-ccf8bbd68db2, 31, Finished, Available, Finished, False)

In [30]:
query = f"""
MERGE INTO delta.`{silver_path}` AS tgt
USING staging_patient AS src
ON tgt.patient_id = src.patient_id
AND tgt.is_current = true

-- 1. EXPIRE OLD RECORD WHEN DATA CHANGES
WHEN MATCHED AND (
    COALESCE(tgt.gender, '') <> COALESCE(src.gender, '') OR
    COALESCE(tgt.family_name, '') <> COALESCE(src.family_name, '') OR
    COALESCE(tgt.given_name, '') <> COALESCE(src.given_name, '') OR
    COALESCE(tgt.last_updated, TIMESTAMP '1900-01-01') <> COALESCE(src.last_updated, TIMESTAMP '1900-01-01')
)
THEN UPDATE SET
    tgt.is_current = false,
    tgt.effective_end_date = current_timestamp()

-- 2. INSERT NEW RECORD IF NOT MATCHED
WHEN NOT MATCHED
THEN INSERT (
    extraction_timestamp,
    load_date,
    patient_id,
    gender,
    family_name,
    given_name,
    last_updated,
    effective_start_date,
    effective_end_date,
    is_current
)
VALUES (
    src.extraction_timestamp,
    src.load_date,
    src.patient_id,
    src.gender,
    src.family_name,
    src.given_name,
    src.last_updated,
    current_timestamp(),
    NULL,
    true
)
"""

query1 = f"""
INSERT INTO delta.`{silver_path}` (
    extraction_timestamp,
    load_date,
    patient_id,
    gender,
    family_name,
    given_name,
    last_updated,
    effective_start_date,
    effective_end_date,
    is_current
)
SELECT 
    src.extraction_timestamp,
    src.load_date,
    src.patient_id,
    src.gender,
    src.family_name,
    src.given_name,
    src.last_updated,
    current_timestamp() AS effective_start_date,
    NULL AS effective_end_date,
    true AS is_current
FROM staging_patient src
JOIN delta.`{silver_path}` tgt
    ON tgt.patient_id = src.patient_id
WHERE tgt.is_current = false
AND (
    COALESCE(tgt.gender, '') <> COALESCE(src.gender, '') OR
    COALESCE(tgt.family_name, '') <> COALESCE(src.family_name, '') OR
    COALESCE(tgt.given_name, '') <> COALESCE(src.given_name, '') OR
    COALESCE(tgt.last_updated, TIMESTAMP '1900-01-01') <> COALESCE(src.last_updated, TIMESTAMP '1900-01-01')
)
AND NOT EXISTS (
    SELECT 1
    FROM delta.`{silver_path}` t2
    WHERE t2.patient_id = src.patient_id
      AND t2.is_current = true
)
"""

spark.sql(query)
spark.sql(query1)

StatementMeta(, b7cd0d21-3cd8-4a68-a843-ccf8bbd68db2, 32, Finished, Available, Finished, False)

DataFrame[]

In [31]:
df = spark.sql(f"DESCRIBE HISTORY delta.`{silver_path}`") \
    .orderBy("version", ascending=False) \
    .limit(2)

display(df)

StatementMeta(, b7cd0d21-3cd8-4a68-a843-ccf8bbd68db2, 33, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, d9757cc9-bb73-4594-a486-d97d814460ad)